# 请求体与数据校验

学习目标：用 Pydantic 模型接收 JSON 请求体，区分字段缺失、空值和类型错误，并写出可观察的输入约束。

前置知识：Python 类、类型标注、异常处理、JSON 与 HTTP 请求体。

适用版本：Python 3.12、Pydantic v2；FastAPI 与兼容依赖按课程环境安装。

环境准备：[环境配置与运行入口](README.md)。

工作目录：本 Notebook 所在的 content/Web与应用开发/FastAPI/。从空内核依次运行；输入均在单元内给出，使用 TestClient 在应用内发送请求。

## 1 用模型接收一条学习记录

客户端把记录标题和学习分钟数放入 JSON 请求体。继承 BaseModel 的类声明数据字段；把这个类写在路由参数的类型位置，FastAPI 就会读取并校验请求体，再把模型实例交给函数。

先只接收两个字段。Record 是这里定义的模型名称；record 是一次请求校验后得到的实例。

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel


class Record(BaseModel):
    title: str
    minutes: int


app = FastAPI()


@app.post("/records")
def receive_record(record: Record):
    return {"title": record.title, "minutes": record.minutes}

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


TestClient 可以直接调用应用，不需要先监听网络端口。post() 的 json 参数接收 Python 数据并将它编码为 JSON 请求体；json() 则读取响应中的 JSON 数据。with 结束时关闭客户端。

In [2]:
with TestClient(app) as client:
    response = client.post("/records", json={"title": "阅读", "minutes": 25})

# 路由收到模型实例，客户端拿到包含两个字段的 JSON 响应。
assert response.status_code == 200
assert response.json() == {"title": "阅读", "minutes": 25}
print(response.status_code, response.json())

200 {'title': '阅读', 'minutes': 25}


缺少必填字段时，请求校验会失败，FastAPI 默认返回 422。错误信息中的 loc 表示出错位置，type 表示错误类别；body 表示请求体。下面只显示这两个字段，方便定位问题。

In [3]:
with TestClient(app) as client:
    response = client.post("/records", json={"title": "阅读"})

error = response.json()["detail"][0]
# 缺失位置指向 body 中的 minutes，而不是路由内部的计算。
assert response.status_code == 422
assert error["loc"] == ["body", "minutes"]
assert error["type"] == "missing"
print(response.status_code, error["loc"], error["type"])

422 ['body', 'minutes'] missing


## 2 用 Field 写清数值和长度约束

类型为整数，还不足以表达“学习分钟数不能为负数”。Field 从 pydantic 导入，用于给模型字段补充约束。这里自行约定标题长度为 1～20 个字符，分钟数为 0～180，包含边界。

| 参数 | 中文名称／含义 |
| --- | --- |
| min_length | 最小长度，包含此长度 |
| max_length | 最大长度，包含此长度 |
| ge | 大于或等于给定值 |
| le | 小于或等于给定值 |

In [4]:
from pydantic import Field


class BoundedRecord(BaseModel):
    title: str = Field(min_length=1, max_length=20)
    minutes: int = Field(ge=0, le=180)


@app.post("/bounded-records")
def receive_bounded_record(record: BoundedRecord):
    return record

对同一条路由只改变 minutes，观察边界两侧的请求。再单独发送空标题，避免把多种失败混在一个输入中。

In [5]:
with TestClient(app) as client:
    for minutes in (-1, 0, 180, 181):
        response = client.post(
            "/bounded-records", json={"title": "阅读", "minutes": minutes}
        )
        # 0 和 180 合法；边界外的两个值应被拒绝。
        expected = 200 if 0 <= minutes <= 180 else 422
        assert response.status_code == expected
        print(minutes, response.status_code)
    empty = client.post("/bounded-records", json={"title": "", "minutes": 25})
    assert empty.status_code == 422
    print("空标题", empty.json()["detail"][0]["type"])

-1 422
0 200
180 200
181 422
空标题 string_too_short


## 3 区分字段缺失与 None

JSON 中的 null 对应 Python 的 None。字段是否允许 None，由类型声明决定；字段能否省略，由默认值决定。这是两个独立问题。

Pydantic v2 中，声明 str | None 并不会自动获得 None 默认值。下面的 required_note 必须出现，但允许值为 null；optional_note 可以省略，省略时采用 None。

In [6]:
class Notes(BaseModel):
    required_note: str | None
    optional_note: str | None = None


@app.post("/notes")
def receive_notes(notes: Notes):
    return notes

先比较省略 required_note 与显式传入 null；再给 optional_note 一个具体字符串。后续单元继续使用本章已经定义的 app 和模型。

In [7]:
cases = [
    ({}, 422),
    ({"required_note": None}, 200),
    ({"required_note": "读完了", "optional_note": None}, 200),
    ({"required_note": None, "optional_note": "明天继续"}, 200),
]
with TestClient(app) as client:
    for data, expected in cases:
        response = client.post("/notes", json=data)
        assert response.status_code == expected
        if expected == 200:
            print(data, "→", response.json())
        else:
            # 空对象的错误是缺少 required_note，不是禁止 None。
            print(data, "→", response.json()["detail"][0]["type"])

{} → missing
{'required_note': None} → {'required_note': None, 'optional_note': None}
{'required_note': '读完了', 'optional_note': None} → {'required_note': '读完了', 'optional_note': None}
{'required_note': None, 'optional_note': '明天继续'} → {'required_note': None, 'optional_note': '明天继续'}


## 4 校验嵌套对象与列表元素

一个字段也可以是另一个模型，或者由特定类型元素组成的列表。FastAPI 会按照嵌套结构继续校验：topic 是 Topic 对象，tags 是字符串列表。

下面要求 topic 中的 name 非空。嵌套错误的位置会继续指向出错字段；列表错误的位置还会包含从 0 开始的元素索引。

In [8]:
class Topic(BaseModel):
    name: str = Field(min_length=1)


class StudySession(BaseModel):
    topic: Topic
    tags: list[str]


@app.post("/sessions")
def receive_session(session: StudySession):
    return session

每次调用都提供完整请求体，只改变要观察的嵌套值。

In [9]:
with TestClient(app) as client:
    valid = client.post(
        "/sessions", json={"topic": {"name": "HTTP"}, "tags": ["复习"]}
    )
    bad_name = client.post(
        "/sessions", json={"topic": {"name": ""}, "tags": ["复习"]}
    )
    bad_tag = client.post(
        "/sessions", json={"topic": {"name": "HTTP"}, "tags": [False]}
    )

assert valid.status_code == 200
assert bad_name.status_code == bad_tag.status_code == 422
assert bad_name.json()["detail"][0]["loc"] == ["body", "topic", "name"]
assert bad_tag.json()["detail"][0]["loc"] == ["body", "tags", 0]
print(valid.json())
print("对象字段：", bad_name.json()["detail"][0]["loc"])
print("列表元素：", bad_tag.json()["detail"][0]["loc"])

{'topic': {'name': 'HTTP'}, 'tags': ['复习']}


对象字段： ['body', 'topic', 'name']
列表元素： ['body', 'tags', 0]


## 5 用字段校验器表达自定规则

长度限制不会自动判断标题是否全为空格。field_validator 可以补充这样的规则。mode="after" 表示先完成 Pydantic 对该字段的类型处理，再把结果传入校验函数。

本例先去掉标题两端的空白，再拒绝空字符串。校验器用 ValueError 表达校验失败，正常路径必须返回最终字段值；这里 cls 是由 classmethod 接收的模型类。

In [10]:
from pydantic import field_validator


class TitledRecord(BaseModel):
    title: str

    @field_validator("title", mode="after")
    @classmethod
    def clean_title(cls, value: str) -> str:
        cleaned = value.strip()
        if not cleaned:
            raise ValueError("标题不能只包含空白")
        return cleaned


@app.post("/titled-records")
def receive_titled_record(record: TitledRecord):
    return record

观察合法标题的调整和纯空白标题的拒绝，确认校验器既可以检查输入，也可以返回处理后的值。

In [11]:
with TestClient(app) as client:
    valid = client.post("/titled-records", json={"title": "  阅读  "})
    invalid = client.post("/titled-records", json={"title": "   "})

assert valid.json() == {"title": "阅读"}
assert invalid.status_code == 422
assert invalid.json()["detail"][0]["type"] == "value_error"
print(valid.json())
print(invalid.status_code, invalid.json()["detail"][0]["msg"])

{'title': '阅读'}
422 Value error, 标题不能只包含空白


## 6 选择类型转换或严格校验

Pydantic 的默认校验允许部分类型转换，例如把整数形式的字符串 "25" 转成整数 25；无法解析的字符串仍然失败。校验通过并不意味着原始输入本来就有目标类型。

先直接调用 Record.model_validate()。这个方法从 Python 数据创建并校验模型；model_dump() 把模型内容转换成字典，便于观察结果。它们使用的仍是本章开始时定义的 Record。

In [12]:
from pydantic import ValidationError

converted = Record.model_validate({"title": "阅读", "minutes": "25"})
assert converted.minutes == 25
assert type(converted.minutes) is int
print(converted.model_dump(), type(converted.minutes).__name__)

try:
    Record.model_validate({"title": "阅读", "minutes": "二十五"})
except ValidationError as error:
    # 这是刻意提供的反例；核对具体类别后继续执行。
    assert error.errors()[0]["type"] == "int_parsing"
    print(error.errors()[0]["loc"], error.errors()[0]["type"])
else:
    raise AssertionError("不可解析的分钟数应被拒绝")

{'title': '阅读', 'minutes': 25} int
('minutes',) int_parsing


当接口约定 minutes 必须来自 JSON 数字时，可以在字段上设置 strict=True。这个整数示例会拒绝字符串 "25"。也可在模型配置中设置 ConfigDict(strict=True)，或在一次 model_validate() 调用中传 strict=True。

严格模式的具体规则取决于字段类型与输入方式。例如直接校验 JSON 时，日期类型在严格模式下仍可接收某些字符串；不要把本例推广为所有类型都完全禁止转换。

In [13]:
class StrictMinutes(BaseModel):
    minutes: int = Field(strict=True)


@app.post("/strict-minutes")
def receive_strict_minutes(data: StrictMinutes):
    return data


with TestClient(app) as client:
    numeric = client.post("/strict-minutes", json={"minutes": 25})
    text = client.post("/strict-minutes", json={"minutes": "25"})

assert numeric.status_code == 200
assert text.status_code == 422
assert text.json()["detail"][0]["type"] == "int_type"
print("JSON 数字：", numeric.json())
print("JSON 字符串：", text.status_code, text.json()["detail"][0]["type"])

JSON 数字： {'minutes': 25}
JSON 字符串： 422 int_type


## 7 决定如何处理额外字段

请求中可能出现模型没有声明的字段。model_config 中的 extra 决定处理方式；ConfigDict 用来书写模型配置。

| extra 的值 | 中文名称／含义 |
| --- | --- |
| ignore | 忽略额外字段，默认行为 |
| forbid | 拒绝额外字段 |
| allow | 接收并保留额外字段 |

下面沿用 Record 的两个字段，只改变额外字段策略。allow 默认不会为未声明的字段附加具体类型约束，不能把“已接收”当作“已经按业务类型检查”。

In [14]:
from pydantic import ConfigDict


class ClosedRecord(Record):
    model_config = ConfigDict(extra="forbid")


class OpenRecord(Record):
    model_config = ConfigDict(extra="allow")


data = {"title": "阅读", "minutes": 25, "source": "paper"}
ignored = Record.model_validate(data).model_dump()
allowed = OpenRecord.model_validate(data).model_dump()
assert "source" not in ignored
assert allowed["source"] == "paper"
print("ignore：", ignored)
print("allow：", allowed)

ignore： {'title': '阅读', 'minutes': 25}
allow： {'title': '阅读', 'minutes': 25, 'source': 'paper'}


把 forbid 策略用于路由，额外字段就会成为客户端可定位的请求校验错误。

In [15]:
@app.post("/closed-records")
def receive_closed_record(record: ClosedRecord):
    return record


with TestClient(app) as client:
    response = client.post("/closed-records", json=data)

error = response.json()["detail"][0]
assert response.status_code == 422
assert error["loc"] == ["body", "source"]
assert error["type"] == "extra_forbidden"
print(response.status_code, error["loc"], error["type"])

422 ['body', 'source'] extra_forbidden


## 8 类型标注与运行时校验各自做什么

普通 Python 函数上的类型标注可以供编辑器和静态类型检查器使用，Python 运行时不会仅因为标注就检查参数。Pydantic 则主动读取模型声明并执行运行时校验；FastAPI 在请求边界调用这类处理。

下面刻意把字符串传给标注为 int 的普通函数，对照 Pydantic 的模型结果。这个反例只用来观察运行时行为，不代表推荐违反类型标注。

In [16]:
def keep_minutes(minutes: int) -> int:
    return minutes


plain_value = keep_minutes("25")
model_value = Record(title="阅读", minutes="25").minutes
# 普通函数原样返回字符串；Pydantic 默认校验把该输入转为整数。
assert type(plain_value) is str
assert type(model_value) is int
print("普通函数：", repr(plain_value), type(plain_value).__name__)
print("模型字段：", repr(model_value), type(model_value).__name__)

普通函数： '25' str
模型字段： 25 int


## 本章小结

（1）模型参数让 FastAPI 接收并校验 JSON 请求体；Field 补充数值和长度条件，校验器补充自定规则。

（2）字段必填与允许 None 分别由默认值和类型声明决定；嵌套模型的错误位置能继续指向子字段和列表元素。

（3）类型转换、严格校验与 extra 策略决定输入边界。普通类型标注本身不执行这些检查。

自查：如果客户端多传了一个字段、把数字写成字符串，或者省略了允许 None 的字段，你能分别指出由哪项声明决定结果吗？

## 练习

1. 新增一个接收 minutes 的模型和路由，约定它是 5～120 的整数，包含边界。分别发送 4、5、120、121，断言中间两个请求返回 200，另外两个返回 422。
2. 新增评分模型：rating 必填但允许 None，comment 可以省略且默认 None。发送缺失 rating、rating 为 None，以及有 comment 的完整输入，核对 422 与 200，并检查省略 comment 后的响应值。
3. 新增含 topic 嵌套对象的模型；topic.name 去掉两端空白后不能为空，外层拒绝额外字段。检查正常标题被整理、空白标题返回 422、额外字段返回 422，并断言两种错误分别指向正确位置。

提示：第 1 题使用 Field；第 2 题分别决定联合类型与默认值；第 3 题把字段校验器放在声明 name 的嵌套模型中，把 extra 配置放在外层模型中。

## 参考与引用来源

1. **FastAPI 官方文档**：[Request Body](https://fastapi.tiangolo.com/tutorial/body/)，定位 Create your data model、Declare it as a parameter、Results；[Body - Fields](https://fastapi.tiangolo.com/tutorial/body-fields/)，定位 Import Field、Declare model attributes；[Body - Nested Models](https://fastapi.tiangolo.com/tutorial/body-nested-models/)，定位 Nested Models、List fields；[Testing](https://fastapi.tiangolo.com/tutorial/testing/)，定位 Using TestClient。支持请求体识别、字段与嵌套校验、应用内请求调用。
2. **Pydantic v2 官方文档**：[Models](https://pydantic.dev/docs/validation/latest/concepts/models/)，定位 Basic model usage、Model methods and properties、Data conversion、Extra data、Nested models；[Fields](https://pydantic.dev/docs/validation/latest/concepts/fields/)，定位 Default values、Field constraints；[Validators](https://pydantic.dev/docs/validation/latest/concepts/validators/)，定位 Field validators 中的 after 校验器；[Strict Mode](https://pydantic.dev/docs/validation/latest/concepts/strict_mode/)，定位 Overview、As a validation parameter、At the field level、As a configuration value；[Conversion Table](https://pydantic.dev/docs/validation/latest/concepts/conversion_table/)，定位 int 与 str 的输入转换条件；[Error Handling](https://pydantic.dev/docs/validation/latest/errors/errors/)，定位 ValidationError、errors() 与 loc。支持本章模型行为与错误定位；默认值采用 v2 规则。
3. **Python 3.12 官方文档**：[typing](https://docs.python.org/3.12/library/typing.html)，定位开篇 Note，说明运行时不强制执行函数和变量的类型标注。
4. **GitHub 上的 FastAPI 官方源码**：[0.141.1 的 exception_handlers.py](https://github.com/fastapi/fastapi/blob/0.141.1/fastapi/exception_handlers.py#L20-L26)，定位 request_validation_exception_handler，核对默认请求校验错误使用 422 和 detail 字段。